# Notebook 05 — Conclusion, Limitations & Future Work

This final notebook steps back from the details and reflects on the **Vuln47** project as a whole: what was set out to do, what was built, what the design buys, and — just as importantly — where it falls short and how it could grow.

It closes the loop opened in **Notebook 00**: vulnerability detection was posed as supervised binary classification over a *graph* representation of code, a GNN was argued for, and then (Notebooks 01–04) the data was explored, the preprocessing pipeline was built, the model was trained, and it was evaluated honestly under class imbalance.

**Structure of this notebook:**
1. What was set out to do (recap of the problem)
2. What was built (the end-to-end system)
3. Why the design is the way it is (key decisions & trade-offs)
4. Limitations — an honest accounting
5. Future work — concrete next steps
6. Closing remarks

## 1. What Was Set Out To Do

> **Given the source of a single C/C++ function, predict whether it contains a security vulnerability.**

This is a **supervised binary classification** problem ($1$ = vulnerable, $0$ = safe), and it matters because memory-safety bugs in C/C++ — a missing bounds check, an off-by-one, a use-after-free — turn into remotely exploitable holes (Heartbleed, countless kernel CVEs). At the scale of real codebases, manual review cannot keep up, so an automated **early-warning filter** that triages which functions deserve human attention is genuinely useful.

Two properties of the problem shaped every downstream decision:

- **Code is structured, not linear.** Two functions can look almost identical token-by-token yet differ in the one *structural* relationship (a guard wrapping a dangerous call) that makes one safe and the other exploitable. This motivated a **graph** representation over a flat token sequence.
- **Vulnerabilities are rare (~2.2–2.8%).** A trivial "always safe" classifier scores ~97% accuracy while catching *zero* bugs. This forced a rejection of accuracy in favour of **F1 / Precision / Recall / PR-AUC**, and handling the imbalance with a **class-weighted loss**.

## 2. What Was Built — The End-to-End System

The project is a single reusable Python package (`source/`) plus a sequence of notebooks that document and exercise it. The full pipeline is:

$$ \text{C code} \xrightarrow{\text{tree-sitter}} \text{AST graph} \xrightarrow{\text{node features}} \text{PyG }\texttt{Data} \xrightarrow{\text{GIN + pooling}} \hat{y} $$

Every arrow is a self-contained, testable component, split along a consistent **OOP architecture** — an abstract generic base at each level, with concrete implementations in nested sub-packages.

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# the two package roots the whole project is built from
for pkg in ["source/preprocessing", "source/training"]:
    print(pkg)
    root = os.path.join(PROJECT_ROOT, pkg)
    for dirpath, _, files in os.walk(root):
        if "__pycache__" in dirpath:
            continue
        pys = sorted(f for f in files if f.endswith(".py"))
        if not pys:
            continue
        rel = os.path.relpath(dirpath, PROJECT_ROOT)
        depth = rel.count(os.sep)
        print("  " * depth + f"{os.path.basename(dirpath)}/")
        for f in pys:
            print("  " * (depth + 1) + f)
    print()

source/preprocessing
  preprocessing/
    __init__.py
    domain_representator.py
    explorer.py
    fetcher.py
    loader.py
    preprocessing_pipeline.py
    data_exploring/
      __init__.py
      data_explorer.py
    data_preprocessing/
      __init__.py
      data_preprocessing_pipeline.py
      data_loading/
        __init__.py
        data_loader.py
      data_representation/
        __init__.py
        data_graph_representation/
          __init__.py
          code_graph.py
          code_graph_config.py
          code_graph_representator.py
        data_node_representation/
          __init__.py
          code_node.py
          code_node_config.py
          code_node_representator.py
      data_fetching/
        __init__.py
        data_fetcher.py
        data_fetcher_config.py

source/training
  training/
    __init__.py
    batch_loader.py
    evaluator.py
    trainer.py
    training_pipeline.py
    model_training/
      __init__.py
      model_trainer.py
      model_traini

### The two halves, side by side

| Layer | Preprocessing (`source/preprocessing`) | Training (`source/training`) |
|---|---|---|
| **Abstract bases** | `Loader`, `DomainRepresentator`, `PreprocessingPipeline`, `Explorer` | `Trainer`, `Evaluator`, `BatchLoader`, `TrainingPipeline` |
| **What it does** | raw JSONL → AST graph → feature tensors → PyG `Data` | `Data` batches → trained GIN → honest metrics |
| **Concrete impls** | `CodeGraphRepresentator`, `CodeNodeRepresentator`, `DataPreprocessingPipeline`, `DataLoader` | `GraphBatchLoader`, `ModelTrainer`, `MetricsEvaluator`, `ModelTrainingPipeline` |

The **model** itself lives in `source/vuln47_gnn_model.py` (`Vuln47GNN` + `build_model`). The cell below confirms it builds and reports its size.

In [2]:
from source.vuln47_gnn_model import build_model
from source.preprocessing.data_preprocessing.data_representation.data_node_representation.code_node_representator import (
    CodeNodeRepresentator,
)

vocab = CodeNodeRepresentator.load_or_build_vocab()
model = build_model(vocab)
n_params = sum(p.numel() for p in model.parameters())

print(f"Vuln47GNN — {len(vocab)} AST node types, {n_params:,} trainable parameters")
print("A deliberately small model: the structural inductive bias of the GNN")
print("does the heavy lifting, so we do not need a huge parameter count.")

Vuln47GNN — 112 AST node types, 182,406 trainable parameters
A deliberately small model: the structural inductive bias of the GNN
does the heavy lifting, so we do not need a huge parameter count.


## 3. Why the Design Is the Way It Is — Key Decisions & Trade-offs

Every non-trivial choice in this project was a trade-off. The most important ones:

### 3.1 Graph (AST) over token sequence
- **Why:** the AST makes structural relationships (a `call_expression` nested *inside* an `if_statement` guard) **explicit and local**, which is exactly the signal that separates a safe copy from an overflow.
- **Trade-off:** the pipeline starts with the AST only — always available, cheap to parse — and is *deliberately designed so richer edges (control-flow, data-flow, à la Devign) can be added later* without touching the model.

### 3.2 GIN over other GNNs
- **Why:** the Graph Isomorphism Network is provably as expressive as the Weisfeiler–Lehman test — the most powerful class of message-passing GNNs at *distinguishing structure*. Since safe vs. vulnerable often differ by subtle structural patterns, expressiveness is worth paying for.
- **Trade-off:** GIN is more sensitive to hyperparameters than a simpler GCN; this is mitigated with BatchNorm, dropout, and **residual connections** for stable training.

### 3.3 Class-weighted loss over resampling
- **Why:** with a ~1:35 imbalance, mistakes on the rare *vulnerable* class are weighted ~35× more heavily (`weight = [1, n_safe/n_vuln]`) inside `CrossEntropyLoss`. This keeps the *natural* data distribution (no synthetic duplication, no discarding of real safe functions).
- **Trade-off:** weighting raises recall at some cost to precision; the decision threshold and PR-AUC allow that curve to be navigated honestly.

### 3.4 Strict OOP mirror between preprocessing and training
- **Why:** one convention (abstract `Generic` base + nested concrete sub-packages + full `source.` imports) means the whole codebase reads the same way top to bottom — easy to extend, easy to review.
- **Trade-off:** more files and boilerplate than a flat script, in exchange for clarity and testability.

## 4. Limitations — An Honest Accounting

Following the spirit of ReVeal and PrimeVul (which warned against over-claiming), this section is explicit about what the project **cannot** do:

- **AST-only structure.** Only syntactic (parent→child) edges are currently used. Many vulnerabilities are fundamentally about **data flow** (a tainted value reaching a sink) or **control flow** across branches — relationships the bare AST does not encode directly. This is the single biggest limiter on achievable recall.
- **Function-level granularity.** One function is classified in isolation. Real vulnerabilities often span **multiple functions** (a caller fails to validate before calling a callee). Inter-procedural bugs remain invisible to the model.
- **Label noise & realism ceiling.** PrimeVul is carefully curated, but commit-based labelling is imperfect: a "fixing" commit may touch functions that were not themselves the root cause. Even the dataset's authors report that strong models struggle here — the realistic setting has a hard ceiling.
- **Dataset skew toward the Linux kernel.** The training distribution is dominated by kernel-style C (Notebook 01). Generalization to very different codebases (embedded, application C++, etc.) is unverified.
- **Hand-crafted token features are shallow.** The 5 token features (dangerous-function flag, size-like name, etc.) encode useful priors but are coarse; they do not capture semantics like *what value* flows into a call.
- **Binary, not localizing.** The model answers "is this function vulnerable?" but not "*where* / *why*." A reviewer still has to find the specific line.

## 5. Future Work — Concrete Next Steps

The architecture was built to make each of these an *extension*, not a rewrite:

1. **Add control-flow and data-flow edges.** The `CodeGraph` dataclass already carries an `edges` list; adding typed edges (CFG/DFG) and giving the GIN edge-type awareness is the highest-value improvement, directly targeting the AST-only limitation above.
2. **Richer node features / learned token embeddings.** Replace (or augment) the 5 hand-crafted token features with subword or code-token embeddings so the model can learn token semantics instead of relying on a fixed dangerous-function list.
3. **Inter-procedural graphs.** Move from a single function to a small call-graph neighbourhood, so cross-function bugs become visible.
4. **Better imbalance handling.** Compare the class-weighted loss against focal loss and threshold-calibration on the PR curve; report which best trades precision for recall on the *test* split.
5. **Explainability.** Add GNN attribution (e.g. GNNExplainer / attention) to highlight the subgraph responsible for a positive prediction — turning the tool from a flag into a *pointer*.
6. **Stronger baselines.** Compare against a token-sequence model (BiLSTM, à la VulDeePecker) and a code language model, on the *same* PrimeVul splits and the *same* metrics, to quantify how much the graph structure actually buys.

## 6. Closing Remarks

Vuln47 delivers a **complete, reproducible pipeline** for graph-based vulnerability detection: from raw C source, through a tree-sitter AST and hand-designed node features, to a GIN classifier trained with imbalance-aware loss and evaluated with the metrics that actually matter for a rare positive class.

The engineering contribution is a **clean, symmetric OOP codebase** — preprocessing and training built on the same abstract-base + nested-implementation pattern — designed so its most important limitation (AST-only structure) can be lifted by *adding* edge types rather than rebuilding. The scientific stance, inherited from ReVeal and PrimeVul, is one of **honest evaluation**: the flattering accuracy number is not reported, and the model is instead measured where it is hard.

There is real headroom left — richer graphs, inter-procedural context, explainability — but the foundation is in place, and every future step has a clear home in the architecture.